# Tutorial: Palette catalog summaries

## Audience
- Python learners who want to summarize a small CSV catalog with only the standard library.

## Prerequisites
- Basic Python: functions, lists, dictionaries and sets.
- The catalog file `/inputs/colors.csv` (UTF-8, with a header row).

## Learning goals
- Read a CSV by header names with `csv.DictReader`.
- Normalize text values (strip whitespace, uppercase hex colors).
- Group rows and sort the groups with a compound sort key.


## Outline

1. Setup
2. Define `summarize_palettes(csv_path)`
3. Summarize the real catalog
4. Exercises


In [ ]:
# Setup: standard library only
from __future__ import annotations

import csv
from collections import defaultdict

CATALOG_PATH = '/inputs/colors.csv'


## Step 1 - Group rows by normalized primary color

Each row's `Primary (Hex)` is stripped and uppercased so `#2563eb ` and `#2563EB` land in the same group.
Every row counts (duplicates included); product types are stripped, de-duplicated and sorted.
Groups are ordered by count (descending), then by primary color (ascending). A header-only file gives `[]`.


In [ ]:
def summarize_palettes(csv_path):
    """Group catalog rows by normalized primary color."""
    counts = defaultdict(int)
    product_types = defaultdict(set)
    with open(csv_path, encoding='utf-8', newline='') as handle:
        for row in csv.DictReader(handle):
            primary = (row.get('Primary (Hex)') or '').strip().upper()
            product = (row.get('Product Type') or '').strip()
            counts[primary] += 1
            product_types[primary].add(product)
    groups = [
        {'primary': primary, 'count': count, 'product_types': sorted(product_types[primary])}
        for primary, count in counts.items()
    ]
    groups.sort(key=lambda group: (-group['count'], group['primary']))
    return groups


## Step 2 - Summarize the real catalog

The next cell runs the function on the catalog and shows the top three groups instead of printing everything.


In [ ]:
palette_summary = summarize_palettes(CATALOG_PATH)
len(palette_summary), palette_summary[:3]


## Exercises

- Predict what happens to a row whose primary color is written in lowercase, then check.
- Write a small CSV with only the header row and confirm the function returns `[]`.
- Common mistake: splitting lines on `,` by hand breaks quoted product names such as `"Shop, Inc"`; `csv.DictReader` handles them.


In [ ]:
# Exercise answer scaffold: sanity checks on the summary
with open(CATALOG_PATH, encoding='utf-8', newline='') as handle:
    row_total = sum(1 for _ in csv.DictReader(handle))
assert all(set(group) == {'primary', 'count', 'product_types'} for group in palette_summary)
assert sum(group['count'] for group in palette_summary) == row_total
palette_summary[0]
